# 05 · 超时预算、重试与熔断

目标：让 retry 服从总 context budget，并通过可注入时钟观察 Closed → Open → HalfOpen。对应文档：`05_large_systems/5.4-high-concurrency-and-ha.md`。

## 从 AbortController 迁移

`context.Context` 把取消和 deadline 沿调用链传递。服务端还要组合 retry、breaker、queue 和 fallback，避免每层独立重试导致尝试次数相乘。

In [ ]:
import (
    "context"
    "errors"
    "fmt"
    "time"
)

func Retry(ctx context.Context, maxAttempts int, call func(attempt int) error) error {
    var lastErr error
    for attempt := 1; attempt <= maxAttempts; attempt++ {
        if err := ctx.Err(); err != nil {
            return fmt.Errorf("retry budget exhausted: %w", err)
        }
        lastErr = call(attempt)
        if lastErr == nil {
            return nil
        }
    }
    return fmt.Errorf("all attempts failed: %w", lastErr)
}

type BreakerState string

const (
    BreakerClosed   BreakerState = "closed"
    BreakerOpen     BreakerState = "open"
    BreakerHalfOpen BreakerState = "half-open"
)

type TimedBreaker struct {
    State       BreakerState
    Failures    int
    Threshold   int
    Cooldown    time.Duration
    OpenedAt    time.Time
    Now         func() time.Time
}

func NewTimedBreaker(threshold int, cooldown time.Duration, now func() time.Time) *TimedBreaker {
    return &TimedBreaker{State: BreakerClosed, Threshold: threshold, Cooldown: cooldown, Now: now}
}

func (b *TimedBreaker) Allow() bool {
    if b.State == BreakerOpen && b.Now().Sub(b.OpenedAt) >= b.Cooldown {
        b.State = BreakerHalfOpen
    }
    return b.State != BreakerOpen
}

func (b *TimedBreaker) Record(err error) {
    if err == nil {
        b.State = BreakerClosed
        b.Failures = 0
        return
    }
    b.Failures++
    if b.State == BreakerHalfOpen || b.Failures >= b.Threshold {
        b.State = BreakerOpen
        b.OpenedAt = b.Now()
    }
}

## 实验一：共享重试预算

前两次调用返回临时错误，第三次成功。先预测实际 attempt 数。随后把 timeout 缩短，观察总 context 如何终止整个策略。

In [ ]:
%%
retryCtx, cancelRetry := context.WithTimeout(context.Background(), time.Second)
defer cancelRetry()

retryErr := Retry(retryCtx, 4, func(attempt int) error {
    fmt.Println("attempt", attempt)
    if attempt < 3 {
        return errors.New("temporary provider failure")
    }
    return nil
})
fmt.Println("retry result:", retryErr)

## 实验二：熔断状态机

连续两次失败后进入 Open；注入 fake clock，无需 sleep 就能推进 cooldown。HalfOpen 探测成功后回到 Closed。

In [ ]:
%%
fakeNow := time.Date(2026, 7, 16, 0, 0, 0, 0, time.UTC)
breaker := NewTimedBreaker(2, 30*time.Second, func() time.Time { return fakeNow })

breaker.Record(errors.New("timeout"))
fmt.Printf("after failure 1: state=%s allow=%t\n", breaker.State, breaker.Allow())
breaker.Record(errors.New("timeout"))
fmt.Printf("after failure 2: state=%s allow=%t\n", breaker.State, breaker.Allow())

fakeNow = fakeNow.Add(31 * time.Second)
fmt.Printf("after cooldown: state-before=%s allow=%t state-after=%s\n", BreakerOpen, breaker.Allow(), breaker.State)
breaker.Record(nil)
fmt.Printf("after probe success: state=%s allow=%t\n", breaker.State, breaker.Allow())

## 组合规则

生产代码中 retry、fallback 和 breaker 必须共享总 attempt/deadline budget。客户端 4xx 通常不应计入 provider breaker；半开探测还需要限制并发与加入 jitter。

In [ ]:
!*cd "$GO_LEARNING_CAPSTONE" && go test ./internal/gateway/provider -run 'TestBreakerOpensAfterRetryableFailures|TestTimedBreakerMovesToHalfOpenAfterCooldown|TestQueueRejectsWhenFull|TestBoundedClientRejectsWhenInFlightFull' -v

## 动手挑战

1. 让 Retry 只重试可恢复错误。
2. 为 backoff 增加 jitter，但让测试仍然确定。
3. 限制 HalfOpen 同时只允许一个探测请求。

## 复盘

- queue capacity 与 max in-flight 分别保护什么？
- breaker 应按 provider、endpoint 还是全局划分？
- 为什么 liveness 不应因一个可 fallback provider 故障而失败？